In [1]:
%pip install tqdm sentence_transformers osmnx networkx ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import pickle
import osmnx as ox
import networkx as nx
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from eval.evaluation import prompt_based_route_evaluator
from routing.keyword_router import KeywordRouter
from routing.synset import OSMSemanticBridge

In [4]:
tag_schema = {
    "continuous": {
        "maxspeed_imputed": {"min": 10, "max": 65, "unit": "mph"},
        "lanes": {"min": 1, "max": 6},
        "length": {"min": 2, "max": 6845}
    },
    "discrete": {
        "highway": ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link',
                    'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
        "access":   ["yes", "no"],
        "bridge":   ["yes", "viaduct"],
        "junction": ["roundabout", "jughandle"],
        "ref":      ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'I 91', 'I 90', 'MA 116']
    }
}

In [5]:
with open("research/pioneer_valley_v2.pkl", "rb") as f:
    G = pickle.load(f)

with open("research/synthetic_dataset.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

# Bridge is still needed for the evaluator's constraint_satisfaction metric
st_model = SentenceTransformer('all-MiniLM-L6-v2')
bridge = OSMSemanticBridge(tag_schema, st_model, threshold=0.80)

router = KeywordRouter(G)

gen_routes, gen_prompts = [], []

for item in tqdm(data[100:200]):
    try:
        s_pt = ox.geocoder.geocode(f"{item['start']}, MA, USA")
        e_pt = ox.geocoder.geocode(f"{item['end']}, MA, USA")

        s_node = ox.distance.nearest_nodes(G, X=s_pt[1], Y=s_pt[0])
        e_node = ox.distance.nearest_nodes(G, X=e_pt[1], Y=e_pt[0])

        if s_node == e_node: continue
        if not nx.has_path(G, s_node, e_node): continue

        route = router.find_route(s_node, e_node, item['prompt'])
        if route:
            gen_routes.append(route)
            gen_prompts.append(item['prompt'])

    except Exception as e:
        print(f"Failed on item {item['id']}: {e}")
        continue

if gen_routes:
    print(f"\nSuccessfully generated {len(gen_routes)} routes. Evaluating...")
    evaluator = prompt_based_route_evaluator(G, gen_prompts, gen_routes, bridge)
    results = evaluator.evaluate_method()

    print("\n--- Keyword Matching Metrics ---")
    for metric, value in results.items():
        print(f"{metric:30}: {value:.4f}")
else:
    print("No valid routes generated.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bridge: Building Schema-Grounded Index for categories: ['highway', 'access', 'bridge', 'junction', 'ref']


100%|██████████| 100/100 [05:50<00:00,  3.50s/it]


Successfully generated 100 routes. Evaluating...


...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Keyword Matching Metrics ---
avg_path_validity             : 1.0000
avg_deviation_penalty         : 1.1895
avg_constraint_satisfaction   : 0.4482
avg_semantic_alignment_f1     : 0.7902
